In [1]:
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace
from scipy.ndimage import gaussian_filter1d
from astropy.stats import biweight_location
from astropy.table import Table, hstack, vstack

import run_config
import utils_lya_halo

from run_config import cfg, smoke
from utils_lya_halo import (run_extract, run_stack, measure, plotting, core, stack, multicat, analysis,
validation)
from utils_lya_halo import guide, pipeline_map, check_guide
from utils_lya_halo.io import read_galaxy_fits, apply_finite_cut

In [2]:
check_guide()

╔══ GUIDE AUDIT ═══════════════════════════════════════════════════════════════════════════════╗
╚══════════════════════════════════════════════════════════════════════════════════════════════╝
  • 3 public function(s) not in the guide (add an entry or list in _BACKEND):
      - analysis.plot_flux_profile_fit
      - validation.combined_bin_significance
      - validation.pool_bins_and_bootstrap


{'missing': [],
 'new': ['analysis.plot_flux_profile_fit',
  'validation.combined_bin_significance',
  'validation.pool_bins_and_bootstrap']}

In [3]:
guide()

╔══ LYAHALO · FIELD GUIDE ═════════════════════════════════════════════════════════════════════╗
║                                                                                              ║
║   pick a section, e.g.  guide("measure")                                                     ║
║                                                                                              ║
║   ► 1  EXTRACTION (18)  Stage 1-2: turn the fiber data + catalog into the galaxy FITS and th ║
║   ► 2  MEASURE    (22)  Stage 3: centroids, integrated flux, asymmetry, and the galaxy-boots ║
║   ► 3  PLOTTING   (27)  Figures for stacks, radial profiles, and per-run diagnostics.        ║
║   ► 4  VALIDATION (39)  Prove the centroid is real: nulls, robustness sweeps, error cross-ch ║
║   ► 5  SAMPLE     (21)  Slice, split, and match the sample -- pure Stage-2, no re-extraction ║
║   ► 6  MISC       (31)  Science extras and separate tracks: virial conversions, LSF/intrinsi ║
║                             

In [4]:
guide('MISC')

╔══ MISC ══════════════════════════════════════════════════════════════════════════════════════╗
║   Science extras and separate tracks: virial conversions, LSF/intrinsic profile, star-PSF    ║
║   subsections: Virial conversions · LSF / intrinsic profile · Star-PSF & line profiles · C   ║
╚══════════════════════════════════════════════════════════════════════════════════════════════╝
── Virial conversions ────────────────────────────────────────────────────────────────────────
  virial_to_kpc_bins
      Convert R/Rvir bin edges to kpc for ONE galaxy's mass & z (Moster+2013 -> R200c). The
      per-galaxy mapping extraction uses; call it to see a galaxy's physical bin edges.
  virial_to_angular_bins
      Convert R/Rvir bin edges to arcsec for one galaxy's mass & z. Useful for overlaying
      bins on imaging.
  physical_kpc_to_arcsec
      Angular size (arcsec) of a physical size (kpc) at redshift z (Planck18). The basic
      cosmology conversion for annotating plots.
  estimate_M200

In [5]:
guide(search="centroid")     # grep names + purposes across everything

╔══ SEARCH · 'centroid'  (32 hits) ════════════════════════════════════════════════════════════╗
╚══════════════════════════════════════════════════════════════════════════════════════════════╝
  run_measure  [measure/Drivers]
      Stage 3: bootstrap the Lya centroid (+ blue/red side ratio) and the per-pixel stack
      error from the Stage-2 cube. Fast; requires stacks built with keep_cube=True.
  measure_all_bins  [measure/Drivers]
      The config-driven measurement engine run_measure calls: centroid + side-ratio
      bootstrap and stack error for every radial bin. Use directly to override stack_method
      or skip the stack error.
  measure_centroid  [measure/Centroid estimators]
      One entry point for every centroid estimator (dispatch by name), all sharing the same
      return contract. Use this rather than calling a specific estimator unless you need
      that estimator's extra kwargs.
  flux_weighted_centroid  [measure/Centroid estimators]
      Continuum-subtract then 

In [6]:
guide('run_extract')

┌─ run_extract ──────────────────────────────────────────────────────── EXTRACTION / Drivers ┐
│ purpose:  Stage 1: extract every galaxy's binned, background-subtracted spectra and write
│ the galaxy FITS. Slow and I/O-bound -- run it once per config; results are cached per
│ galaxy so re-runs are cheap.
│ module:   utils_lya_halo.pipeline
│ example:
│     path = run_extract(cfg)
└──────────────────────────────────────────────────────────────────────────────────────────────┘


In [7]:
guide('run_stack')

┌─ run_stack ────────────────────────────────────────────────────────── EXTRACTION / Drivers ┐
│ purpose:  Stage 2: load the galaxy FITS and coadd into rest-frame stacks (one per combine
│ method). Fast -- re-run freely. keep_cube=True is required if you will measure or
│ bootstrap afterwards.
│ module:   utils_lya_halo.pipeline
│ example:
│     stacks = run_stack(cfg, path, keep_cube=True)
└──────────────────────────────────────────────────────────────────────────────────────────────┘


In [8]:
guide('run_measure')

┌─ run_measure ─────────────────────────────────────────────────────────── MEASURE / Drivers ┐
│ purpose:  Stage 3: bootstrap the Lya centroid (+ blue/red side ratio) and the per-pixel
│ stack error from the Stage-2 cube. Fast; requires stacks built with keep_cube=True.
│ module:   utils_lya_halo.pipeline
│ example:
│     boot = run_measure(cfg, stacks)
└──────────────────────────────────────────────────────────────────────────────────────────────┘
